# ROGII Wellbore Geology Prediction

## Notebook 01 - Data Understanding




### 1. Import Libraries


In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

### 2. Configure Project Paths

In [2]:
# Project paths
PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"

TRAIN_DIR = RAW_DIR / "train"
TEST_DIR = RAW_DIR / "test"

RESULTS_DIR = PROJECT_ROOT / "results" / "notebook_01"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

### 3. Validate Dataset Structure

In [3]:
print(f"Project Root : {PROJECT_ROOT.resolve()}")
print(f"Train Exists : {TRAIN_DIR.exists()}")
print(f"Test Exists  : {TEST_DIR.exists()}")

Project Root : C:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology
Train Exists : True
Test Exists  : True


### 4. Count Training Files

In [4]:
typewell_files = sorted(TRAIN_DIR.glob("*_typewell.csv"))
horizontal_files = sorted(TRAIN_DIR.glob("*_horizontal_well.csv"))

print(f"Typewell files   : {len(typewell_files)}")
print(f"Horizontal files : {len(horizontal_files)}")

Typewell files   : 773
Horizontal files : 773


### 5. Validate Well Pairs

Well identifiers are recovered by removing the exact filename suffixes `__typewell.csv` and `__horizontal_well.csv`. A simple string replace of `_typewell.csv` is avoided because training filenames use a double underscore and would otherwise leave a trailing underscore in the well ID.

In [5]:
TYPEWELL_SUFFIX = "__typewell.csv"
HORIZONTAL_SUFFIX = "__horizontal_well.csv"


def extract_well_id(filename: str, suffix: str):
    """Return the well ID if the filename ends with the expected suffix."""
    if filename.endswith(suffix):
        well_id = filename[: -len(suffix)]
        if well_id and not well_id.endswith("_"):
            return well_id
    return None


typewell_ids = set()
horizontal_ids = set()
malformed_typewell = []
malformed_horizontal = []

for file in typewell_files:
    well_id = extract_well_id(file.name, TYPEWELL_SUFFIX)
    if well_id is None:
        malformed_typewell.append(file.name)
    else:
        typewell_ids.add(well_id)

for file in horizontal_files:
    well_id = extract_well_id(file.name, HORIZONTAL_SUFFIX)
    if well_id is None:
        malformed_horizontal.append(file.name)
    else:
        horizontal_ids.add(well_id)

complete_pairs = typewell_ids & horizontal_ids
missing_horizontal = typewell_ids - horizontal_ids
missing_typewell = horizontal_ids - typewell_ids

print(f"Complete pairs     : {len(complete_pairs)}")
print(f"Missing horizontal : {len(missing_horizontal)}")
print(f"Missing typewell   : {len(missing_typewell)}")
print(f"Malformed typewell : {len(malformed_typewell)}")
print(f"Malformed horizontal: {len(malformed_horizontal)}")

Complete pairs     : 773
Missing horizontal : 0
Missing typewell   : 0
Malformed typewell : 0
Malformed horizontal: 0


### 6. Well-ID Extraction Validation

This section confirms that horizontal-well and typewell identifiers match after suffix removal, and that no well ID retains a trailing underscore or filename suffix remnant.

In [6]:
well_id_validation = pd.DataFrame(
    [
        {
            "Metric": "Number of horizontal wells",
            "Count": len(horizontal_ids),
        },
        {
            "Metric": "Number of typewells",
            "Count": len(typewell_ids),
        },
        {
            "Metric": "Number of matched pairs",
            "Count": len(complete_pairs),
        },
        {
            "Metric": "Number of unmatched files",
            "Count": (
                len(missing_horizontal)
                + len(missing_typewell)
                + len(malformed_typewell)
                + len(malformed_horizontal)
            ),
        },
        {
            "Metric": "Number of malformed IDs",
            "Count": len(malformed_typewell) + len(malformed_horizontal),
        },
        {
            "Metric": "IDs with trailing underscore",
            "Count": sum(
                well_id.endswith("_")
                for well_id in (typewell_ids | horizontal_ids)
            ),
        },
    ]
)

display(well_id_validation)

validation_path = TABLES_DIR / "well_id_validation.csv"
well_id_validation.to_csv(validation_path, index=False)
print(f"Saved: {validation_path.resolve()}")

example_ids = sorted(complete_pairs)[:5]
print("Example matched well IDs:", example_ids)
assert all(not well_id.endswith("_") for well_id in complete_pairs)
assert len(malformed_typewell) == 0
assert len(malformed_horizontal) == 0

,Metric,Count
0,Number of horizontal wells,773
1,Number of typewells,773
2,Number of matched pairs,773
3,Number of unmatched files,0
4,Number of malformed IDs,0
5,IDs with trailing underscore,0


Saved: C:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_01\tables\well_id_validation.csv
Example matched well IDs: ['000d7d20', '00bbac68', '00e12e8b', '015fe0d2', '01869cd4']


**Interpretation.** The validation table summarises pairing integrity after suffix-safe ID extraction. The most important numerical check is that the number of matched pairs equals both the typewell and horizontal counts, with zero unmatched or malformed identifiers. This matters because later notebooks join files by well ID; a trailing underscore would be an avoidable identifier defect rather than a geological property. The limitation of this check is that it validates naming and pairing only, not row-level data quality. The implication is that subsequent analyses can treat well IDs as clean join keys.

### 7. Load a Sample Well Pair

The lexicographically first matched pair is loaded as a worked example. It is not claimed to be statistically representative of the full training set; dataset-wide selection of typical and difficult wells is deferred to Notebook 03.

In [7]:
sample_id = sorted(complete_pairs)[0]

sample_typewell_path = TRAIN_DIR / f"{sample_id}{TYPEWELL_SUFFIX}"
sample_horizontal_path = TRAIN_DIR / f"{sample_id}{HORIZONTAL_SUFFIX}"

typewell_df = pd.read_csv(sample_typewell_path)
horizontal_df = pd.read_csv(sample_horizontal_path)

print("Sample ID:", sample_id)
print("Typewell path:", sample_typewell_path.name)
print("Horizontal path:", sample_horizontal_path.name)
print("Typewell shape:", typewell_df.shape)
print("Horizontal shape:", horizontal_df.shape)
assert not sample_id.endswith("_")
assert "__" in sample_typewell_path.name

Sample ID: 000d7d20
Typewell path: 000d7d20__typewell.csv
Horizontal path: 000d7d20__horizontal_well.csv
Typewell shape: (1296, 3)
Horizontal shape: (5278, 13)


### 8. Validate Prediction Target

`TVT_input` stores the known portion of the target `TVT`. In the region that must be predicted, `TVT_input` is intentionally missing. This missingness is therefore not ordinary accidental missing data, and conventional mean or median imputation of `TVT_input` would be inappropriate. On known rows, `TVT_input` reproduces `TVT`, so exact agreement is expected.

In [8]:
known_tvt_mask = horizontal_df["TVT_input"].notna()
prediction_mask = ~known_tvt_mask

known_rows = int(known_tvt_mask.sum())
prediction_rows = int(prediction_mask.sum())

prediction_start_index = (
    horizontal_df.index[prediction_mask][0]
    if prediction_rows > 0
    else None
)

tvt_input_matches_tvt = np.allclose(
    horizontal_df.loc[known_tvt_mask, "TVT_input"],
    horizontal_df.loc[known_tvt_mask, "TVT"],
)

# Trailing-block check: all known rows precede all prediction rows
clean_trailing_block = False
if prediction_start_index is not None:
    clean_trailing_block = bool(
        known_tvt_mask.iloc[:prediction_start_index].all()
        and prediction_mask.iloc[prediction_start_index:].all()
    )
elif known_rows == len(horizontal_df):
    clean_trailing_block = True

print(f"Known TVT rows              : {known_rows}")
print(f"Rows to predict             : {prediction_rows}")
print(f"Prediction start row        : {prediction_start_index}")
print(f"TVT_input equals TVT        : {tvt_input_matches_tvt}")
print(f"Clean trailing missing block: {clean_trailing_block}")

Known TVT rows              : 1442
Rows to predict             : 3836
Prediction start row        : 1442
TVT_input equals TVT        : True
Clean trailing missing block: True


### 9. Inspect Columns

In [9]:
print("Typewell columns:")
print(typewell_df.columns.tolist())

print("\nHorizontal well columns:")
print(horizontal_df.columns.tolist())

Typewell columns:
['TVT', 'GR', 'Geology']

Horizontal well columns:
['MD', 'X', 'Y', 'Z', 'ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA', 'TVT', 'GR', 'TVT_input']


### 10. Preview the Data

In [10]:
print("Typewell Dataset")
display(typewell_df.head())

print("\nHorizontal Well Dataset")
display(horizontal_df.head())

Typewell Dataset


,TVT,GR,Geology
0,11223.95,126.11,NaN
1,11224.45,128.22,NaN
2,11224.95,128.72,NaN
3,11225.45,128.12,NaN
4,11225.95,125.29,NaN



Horizontal Well Dataset


,MD,X,Y,Z,ANCC,ASTNU,ASTNL,EGFDU,EGFDL,BUDA,TVT,GR,TVT_input
0,11467.0,2983525.16,1069022.09,-9258.57,-9395.81,-9569.86,-9597.64,-9670.99,-9705.96,-9846.35,11236.02,115.692586,11236.02
1,11468.0,2983525.18,1069022.30,-9259.55,-9395.75,-9569.80,-9597.58,-9670.93,-9705.90,-9846.29,11237.05,115.584293,11237.05
2,11469.0,2983525.20,1069022.52,-9260.52,-9395.69,-9569.74,-9597.52,-9670.87,-9705.84,-9846.23,11238.09,135.446960,11238.09
3,11470.0,2983525.22,1069022.73,-9261.50,-9395.64,-9569.69,-9597.47,-9670.82,-9705.79,-9846.18,11239.12,140.401346,11239.12
4,11471.0,2983525.25,1069022.95,-9262.47,-9395.58,-9569.63,-9597.41,-9670.76,-9705.73,-9846.12,11240.15,111.270638,11240.15


### 11. Dataset Information

In [11]:
print("Typewell Information")
typewell_df.info()

print("\n----------------------------------------\n")

print("Horizontal Well Information")
horizontal_df.info()

Typewell Information
<class 'pandas.DataFrame'>
RangeIndex: 1296 entries, 0 to 1295
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   TVT      1296 non-null   float64
 1   GR       1296 non-null   float64
 2   Geology  997 non-null    str    
dtypes: float64(2), str(1)
memory usage: 34.9 KB

----------------------------------------

Horizontal Well Information
<class 'pandas.DataFrame'>
RangeIndex: 5278 entries, 0 to 5277
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   MD         5278 non-null   float64
 1   X          5278 non-null   float64
 2   Y          5278 non-null   float64
 3   Z          5278 non-null   float64
 4   ANCC       5278 non-null   float64
 5   ASTNU      5278 non-null   float64
 6   ASTNL      5278 non-null   float64
 7   EGFDU      5278 non-null   float64
 8   EGFDL      5278 non-null   float64
 9   BUDA       5278 non-null   float6

### 12. Statistical Summary

In [12]:
print("Typewell Summary")
display(typewell_df.describe())

print("\nHorizontal Well Summary")
display(horizontal_df.describe())

Typewell Summary


,TVT,GR
count,1296.000000,1296.000000
mean,11547.700000,83.257639
std,187.133642,26.251321
min,11223.950000,28.660000
25%,11385.825000,64.940000
50%,11547.700000,87.090000
75%,11709.575000,102.802500
max,11871.450000,158.180000



Horizontal Well Summary


,MD,X,Y,Z,ANCC,ASTNU,ASTNL,EGFDU,EGFDL,BUDA,TVT,GR,TVT_input
count,5278.000000,5.278000e+03,5.278000e+03,5278.000000,5278.000000,5278.000000,5278.000000,5278.000000,5278.000000,5278.000000,5278.000000,3020.000000,1442.000000
mean,14105.500000,2.983514e+06,1.071418e+06,-9673.194324,-9330.621821,-9504.671821,-9532.451821,-9605.801821,-9640.771821,-9781.161821,11715.831127,94.436961,11646.417864
std,1523.771691,2.997379e+01,1.503381e+03,80.961903,35.272977,35.272977,35.272977,35.272977,35.272977,35.272977,88.782092,18.627493,148.869554
min,11467.000000,2.983468e+06,1.069022e+06,-9755.610000,-9395.810000,-9569.860000,-9597.640000,-9670.990000,-9705.960000,-9846.350000,11236.020000,31.765827,11236.020000
25%,12786.250000,2.983487e+06,1.070090e+06,-9721.762500,-9362.307500,-9536.357500,-9564.137500,-9637.487500,-9672.457500,-9812.847500,11735.850000,85.068343,11566.195000
50%,14105.500000,2.983514e+06,1.071407e+06,-9689.230000,-9335.975000,-9510.025000,-9537.805000,-9611.155000,-9646.125000,-9786.515000,11742.130000,94.936506,11745.920000
75%,15424.750000,2.983535e+06,1.072723e+06,-9650.035000,-9296.665000,-9470.715000,-9498.495000,-9571.845000,-9606.815000,-9747.205000,11747.257500,103.960888,11747.465000
max,16744.000000,2.983578e+06,1.074041e+06,-9258.570000,-9271.000000,-9445.050000,-9472.830000,-9546.180000,-9581.150000,-9721.540000,11756.120000,217.352257,11756.120000


### 13. Check Missing Values

Missingness in `TVT_input` marks the prediction region. Missingness in `GR` and typewell `Geology` is ordinary incomplete measurement or labelling and should be analysed separately from the intentional `TVT_input` mask.

In [ ]:
print("Typewell Missing Values")
display(typewell_df.isnull().sum())

print("\nHorizontal Missing Values")
display(horizontal_df.isnull().sum())

Typewell Missing Values


TVT          0
GR           0
Geology    299
dtype: int64


Horizontal Missing Values


MD              0
X               0
Y               0
Z               0
ANCC            0
ASTNU           0
ASTNL           0
EGFDU           0
EGFDL           0
BUDA            0
TVT             0
GR           2258
TVT_input    3836
dtype: int64

### 14. Initial Observations

- Successfully loaded a complete training well pair using suffix-safe well IDs.
- The training dataset contains matched typewell and horizontal-well files; the validation table reports matched pairs, unmatched files, and malformed IDs.
- The typewell dataset contains `TVT`, `GR`, and geological formation labels in the `Geology` column.
- The horizontal-well dataset contains trajectory measurements, formation-marker columns, `GR`, `TVT_input`, and the target column `TVT`.
- `TVT_input` contains the known part of `TVT` and becomes missing in the region that must be predicted. That missingness is intentional, not accidental.
- Conventional mean or median imputation of `TVT_input` is not appropriate.
- Exact agreement between `TVT_input` and `TVT` on known rows is expected and is not an independent predictive discovery.
- The complete horizontal-well `TVT` column is available in training for evaluation, but must not be used as a model input feature.
- Ordinary missing values also appear in typewell `Geology` and horizontal `GR`; these should be treated separately from the `TVT_input` prediction mask.
- The sample well used here is a worked example only. Dataset-wide EDA, including criteria-based well selection, continues in Notebooks 02 and 03.

### 15. Dataset Scale Snapshot (Train and Test Counts)

This snapshot reports file- and pair-level scale for training and test folders without reading test targets. Row totals across all wells are deferred to the dataset-wide notebook, which processes every pair.

In [ ]:
test_typewell_files = sorted(TEST_DIR.glob("*_typewell.csv"))
test_horizontal_files = sorted(TEST_DIR.glob("*_horizontal_well.csv"))

test_typewell_ids = set()
test_horizontal_ids = set()
test_malformed = []

for file in test_typewell_files:
    well_id = extract_well_id(file.name, TYPEWELL_SUFFIX)
    if well_id is None:
        test_malformed.append(file.name)
    else:
        test_typewell_ids.add(well_id)

for file in test_horizontal_files:
    well_id = extract_well_id(file.name, HORIZONTAL_SUFFIX)
    if well_id is None:
        test_malformed.append(file.name)
    else:
        test_horizontal_ids.add(well_id)

test_complete_pairs = test_typewell_ids & test_horizontal_ids

dataset_scale_snapshot = pd.DataFrame(
    [
        {"Split": "train", "Metric": "Typewell files", "Value": len(typewell_files)},
        {"Split": "train", "Metric": "Horizontal files", "Value": len(horizontal_files)},
        {"Split": "train", "Metric": "Matched pairs", "Value": len(complete_pairs)},
        {"Split": "test", "Metric": "Typewell files", "Value": len(test_typewell_files)},
        {"Split": "test", "Metric": "Horizontal files", "Value": len(test_horizontal_files)},
        {"Split": "test", "Metric": "Matched pairs", "Value": len(test_complete_pairs)},
        {"Split": "test", "Metric": "Malformed IDs", "Value": len(test_malformed)},
        {
            "Split": "sample_train_well",
            "Metric": "Horizontal rows",
            "Value": len(horizontal_df),
        },
        {
            "Split": "sample_train_well",
            "Metric": "Typewell rows",
            "Value": len(typewell_df),
        },
        {
            "Split": "sample_train_well",
            "Metric": "Prediction-region rows",
            "Value": int(horizontal_df["TVT_input"].isna().sum()),
        },
    ]
)

display(dataset_scale_snapshot)

scale_path = TABLES_DIR / "dataset_scale_snapshot.csv"
dataset_scale_snapshot.to_csv(scale_path, index=False)
print(f"Saved: {scale_path.resolve()}")
print("Test matched well IDs:", sorted(test_complete_pairs))

,Split,Metric,Value
0,train,Typewell files,773
1,train,Horizontal files,773
2,train,Matched pairs,773
3,test,Typewell files,3
4,test,Horizontal files,3
5,test,Matched pairs,3
6,test,Malformed IDs,0
7,sample_train_well,Horizontal rows,5278
8,sample_train_well,Typewell rows,1296
9,sample_train_well,Prediction-region rows,3836


Saved: C:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_01\tables\dataset_scale_snapshot.csv
Test matched well IDs: ['000d7d20', '00bbac68', '00e12e8b']


**Interpretation.** The table contrasts training and test pair counts at file level. The key finding is the large imbalance between many training pairs and only a small number of test pairs. This matters because any later train–test distribution comparison will be provisional. The limitation is that full row totals across all wells are not yet aggregated here. The implication is that Notebook 03 should report complete dataset-scale row counts and avoid strong claims about distribution shift from the test set alone.